# Benchmark YOLO — Smart Traffic Dataset Clean
**PFE Smart Traffic Agadir — Mouad ASSARGUAL**  
**FSA Aït Melloul, Université Ibn Zohr**

---

## Dataset
- **Source** : Bellevue Traffic Video Dataset + Lqliaa Agadir + Emergency v2
- **Annotations** : Manuelles (Roboflow SAM3 + révision humaine)
- **Total** : ~1942 images (Train: 1557 | Val: 231 | Test: 154)
- **Classes** : `bus, car, emergency_vehicle, motorcycle, person, truck`

## Modèles benchmarkés
| Session | Modèle | Batch | Epochs |
|---------|--------|-------|--------|
| 1 | YOLOv8n | 64 | 100 |
| 2 | YOLOv8s | 32 | 100 |
| 3 | YOLOv11n | 64 | 100 |
| 4 | YOLOv11s | 32 | 100 |
| 5 | YOLO26n | 64 | 100 |
| 6 | YOLO26s | 32 | 100 |

## Corrections appliquées
- ✅ Class weighting (loss-aware training)
- ✅ Early stopping (patience=20)
- ✅ Augmentation avancée
- ✅ Ordre classes Roboflow respecté
- ✅ Validation sur test set
- ✅ Sauvegarde Drive automatique

In [ ]:
# ═══════════════════════════════════════════════════
# CELLULE 1 — Environnement + Drive
# ═══════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import torch, os

# Chemins
DRIVE_DIR   = '/content/drive/MyDrive/lasttraining'
WORK_DIR    = '/content/benchmark'
RESULTS_DIR = f'{DRIVE_DIR}/benchmark_results'

os.makedirs(WORK_DIR,    exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Infos GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU  : {gpu_name}')
    print(f'✅ VRAM : {vram_gb:.1f} GB')
else:
    print('❌ GPU non disponible — vérifier Runtime')

print(f'✅ Drive monté')
print(f'✅ Work dir : {WORK_DIR}')
print(f'✅ Results  : {RESULTS_DIR}')

Mounted at /content/drive
❌ GPU non disponible — vérifier Runtime
✅ Drive monté
✅ Work dir : /content/benchmark
✅ Results  : /content/drive/MyDrive/lasttraining/benchmark_results


In [ ]:
# ═══════════════════════════════════════════════════
# CELLULE 2 — Installation des dépendances
# ═══════════════════════════════════════════════════
!pip install ultralytics -q

import ultralytics
print(f'✅ Ultralytics {ultralytics.__version__}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 43.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Ultralytics 8.4.55


In [ ]:
# ═══════════════════════════════════════════════════
# CELLULE 3 — Chargement dataset depuis Drive
# ═══════════════════════════════════════════════════
import zipfile, os, glob

DATASET_DIR = f'{WORK_DIR}/dataset'
ZIP_PATH    = f'{DRIVE_DIR}/dataset_clean_final.zip'

if not os.path.exists(f'{DATASET_DIR}/images/train'):
    print('Extraction dataset_clean_final.zip...')
    if not os.path.exists(ZIP_PATH):
        raise FileNotFoundError(f'dataset_clean_final.zip introuvable dans {DRIVE_DIR}')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(WORK_DIR)
    # Renommer le dossier extrait
    extracted = glob.glob(f'{WORK_DIR}/dataset_clean_merged_bbox')
    if extracted:
        os.rename(extracted[0], DATASET_DIR)
    print('Extraction terminée ✅')
else:
    print('Dataset déjà extrait ✅')

# Statistiques
n_train = len(glob.glob(f'{DATASET_DIR}/images/train/*'))
n_val   = len(glob.glob(f'{DATASET_DIR}/images/val/*'))
n_test  = len(glob.glob(f'{DATASET_DIR}/images/test/*'))
print(f'\nDataset :')
print(f'  Train : {n_train}')
print(f'  Val   : {n_val}')
print(f'  Test  : {n_test}')
print(f'  Total : {n_train + n_val + n_test}')

Extraction dataset_clean_final.zip...
Extraction terminée ✅

Dataset :
  Train : 1457
  Val   : 290
  Test  : 195
  Total : 1942


In [ ]:
# ═══════════════════════════════════════════════════
# CELLULE 4 — Configuration data.yaml
# ═══════════════════════════════════════════════════
import yaml, glob
from collections import Counter
import numpy as np

# ⚠️ Ordre des classes — IDENTIQUE à Roboflow
CLASS_NAMES = ['bus', 'car', 'emergency_vehicle', 'motorcycle', 'person', 'truck']

# Créer data.yaml
data = {
    'path'  : DATASET_DIR,
    'train' : 'images/train',
    'val'   : 'images/val',
    'test'  : 'images/test',
    'nc'    : len(CLASS_NAMES),
    'names' : CLASS_NAMES
}

YAML_OUT = f'{WORK_DIR}/data.yaml'
with open(YAML_OUT, 'w') as f:
    yaml.dump(data, f)

# Distribution des classes
print('Distribution des classes (train) :')
labels = glob.glob(f'{DATASET_DIR}/labels/train/*.txt')
counts = Counter()
for lbl in labels:
    with open(lbl) as f:
        for line in f:
            if line.strip():
                counts[int(line.split()[0])] += 1

total_ann = sum(counts.values())
for i, name in enumerate(CLASS_NAMES):
    c = counts.get(i, 0)
    pct = c/total_ann*100 if total_ann > 0 else 0
    print(f'  {i} - {name:20s} : {c:6d} ({pct:.1f}%)')

print(f'\n✅ YAML → {YAML_OUT}')
print(f'Classes : {CLASS_NAMES}')

Distribution des classes (train) :
  0 - bus                  :    336 (1.6%)
  1 - car                  :  17131 (81.4%)
  2 - emergency_vehicle    :    307 (1.5%)
  3 - motorcycle           :     89 (0.4%)
  4 - person               :   2229 (10.6%)
  5 - truck                :    962 (4.6%)

✅ YAML → /content/benchmark/data.yaml
Classes : ['bus', 'car', 'emergency_vehicle', 'motorcycle', 'person', 'truck']


In [ ]:
# Dans Colab — cellule de diagnostic
import glob
from collections import Counter

labels = glob.glob(f'{DATASET_DIR}/labels/test/*.txt')
counts = Counter()
for lbl in labels:
    with open(lbl) as f:
        for line in f:
            if line.strip():
                counts[int(line.split()[0])] += 1
print(counts)

Counter({1: 1978, 4: 277, 5: 122, 2: 40, 0: 38, 3: 21})


In [ ]:
# ═══════════════════════════════════════════════════
# CELLULE 5 — Calcul des class weights (loss-aware)
# ═══════════════════════════════════════════════════
import numpy as np

# Compter instances par classe
class_counts = [counts.get(i, 1) for i in range(len(CLASS_NAMES))]
total        = sum(class_counts)

# Poids inversement proportionnels à la fréquence
# w_i = total / (nc * count_i)
weights_raw  = [total / (len(CLASS_NAMES) * c) for c in class_counts]

# Normaliser entre 0.1 et 1.0 pour YOLO
max_w        = max(weights_raw)
weights_norm = [round(w / max_w, 4) for w in weights_raw]

print('Class weights (loss-aware training) :')
for i, (name, w) in enumerate(zip(CLASS_NAMES, weights_norm)):
    bar = '█' * int(w * 20)
    print(f'  {i} - {name:20s} : {w:.4f} {bar}')

# cls_pw pour YOLO = poids de la classe la moins fréquente
# On prend la médiane pour éviter les extrêmes
CLS_PW = round(float(np.median(weights_norm)), 4)
print(f'\ncls_pw utilisé : {CLS_PW}')

Class weights (loss-aware training) :
  0 - bus                  : 0.5526 ███████████
  1 - car                  : 0.0106 
  2 - emergency_vehicle    : 0.5250 ██████████
  3 - motorcycle           : 1.0000 ████████████████████
  4 - person               : 0.0758 █
  5 - truck                : 0.1721 ███

cls_pw utilisé : 0.3486


In [ ]:
# ═══════════════════════════════════════════════════
# CELLULE 6 — ⚠️ MODIFIER À CHAQUE SESSION
# ═══════════════════════════════════════════════════

# ┌─────────────────────────────────────────────────┐
# │  Session 1 : YOLOv8n  / yolov8n.pt  / batch=64 │
# │  Session 2 : YOLOv8s  / yolov8s.pt  / batch=32 │
# │  Session 3 : YOLOv11n / yolo11n.pt  / batch=64 │
# │  Session 4 : YOLOv11s / yolo11s.pt  / batch=32 │
# │  Session 5 : YOLO26n  / yolo26n.pt  / batch=64 │
# │  Session 6 : YOLO26s  / yolo26s.pt  / batch=32 │
# └─────────────────────────────────────────────────┘

MODEL_NAME = 'YOLOv8s'
MODEL_ID   = 'yolov8s.pt'
BATCH      = 32
EPOCHS     = 100

print(f'Session : {MODEL_NAME}')
print(f'Weights : {MODEL_ID}')
print(f'Batch   : {BATCH}')
print(f'Epochs  : {EPOCHS}')
print(f'cls_pw  : {CLS_PW}')

Session : YOLOv8s
Weights : yolov8s.pt
Batch   : 32
Epochs  : 100
cls_pw  : 0.3486


In [ ]:
# ═══════════════════════════════════════════════════
# CELLULE 7 — Entraînement
# ═══════════════════════════════════════════════════
from ultralytics import YOLO
import shutil, glob, time, torch

print(f'\n{"═"*60}')
print(f'  Entraînement : {MODEL_NAME}')
print(f'  GPU          : {torch.cuda.get_device_name(0)}')
print(f'  VRAM         : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'  Dataset      : {n_train} train | {n_val} val')
print(f'  Epochs       : {EPOCHS} (patience=20)')
print(f'  cls_pw       : {CLS_PW}')
print(f'{"═"*60}\n')

model = YOLO(MODEL_ID)

# Callback sauvegarde Drive
CKPT_DIR = f'{RESULTS_DIR}/checkpoints/{MODEL_NAME}'
os.makedirs(CKPT_DIR, exist_ok=True)

def save_checkpoint(trainer):
    epoch = trainer.epoch + 1
    if epoch % 10 == 0:
        dst = f'{CKPT_DIR}/epoch{epoch:03d}.pt'
        shutil.copy(trainer.last, dst)
        if os.path.exists(trainer.best):
            shutil.copy(trainer.best, f'{CKPT_DIR}/best_so_far.pt')
        print(f'  💾 Checkpoint epoch {epoch} → Drive')
        # Garder 3 derniers
        ckpts = sorted(glob.glob(f'{CKPT_DIR}/epoch*.pt'))
        for old in ckpts[:-3]:
            os.remove(old)

model.add_callback('on_train_epoch_end', save_checkpoint)

t0 = time.time()
results = model.train(
    data         = YAML_OUT,
    epochs       = EPOCHS,
    imgsz        = 640,
    batch        = BATCH,
    # Optimiseur
    lr0          = 0.005,   # ← changer 0.01 → 0.005
    lrf          = 0.001,   # ← changer 0.01 → 0.001
    momentum     = 0.937,
    weight_decay = 0.0005,
    # Anti-overfitting
    patience     = 50,      # ← changer 20 → 50
    dropout      = 0.0,
    # Class weighting
    cls_pw       = CLS_PW,
    # Augmentation
    augment      = True,
    mosaic       = 1.0,
    mixup        = 0.1,
    fliplr       = 0.5,
    flipud       = 0.0,
    hsv_h        = 0.015,
    hsv_s        = 0.7,
    hsv_v        = 0.4,
    # Configuration
    project      = f'{WORK_DIR}/runs',
    name         = MODEL_NAME,
    exist_ok     = True,
    device       = 0,
    workers      = 4,
    cache        = False,
    verbose      = True,
)
TRAIN_TIME = time.time() - t0
print(f'\n✅ Entraînement terminé en {TRAIN_TIME/3600:.2f}h')


════════════════════════════════════════════════════════════
  Entraînement : YOLOv8s


AssertionError: Torch not compiled with CUDA enabled

In [ ]:
# ═══════════════════════════════════════════════════
# CELLULE 8 — Évaluation sur test set
# ═══════════════════════════════════════════════════
from ultralytics import YOLO
import os, shutil

BEST_PT     = f'{WORK_DIR}/runs/{MODEL_NAME}/weights/best.pt'
model_final = YOLO(BEST_PT)

# Évaluation sur TEST set (pas val)
metrics = model_final.val(
    data   = YAML_OUT,
    device = 0,
    split  = 'test',
)

SIZE_MB = os.path.getsize(BEST_PT) / 1e6
PARAMS  = sum(p.numel() for p in model_final.model.parameters()) / 1e6

print(f'\n{"═"*60}')
print(f'  RÉSULTATS : {MODEL_NAME}')
print(f'{"═"*60}')
print(f'  mAP@50       : {metrics.box.map50*100:.2f}%')
print(f'  mAP@50-95    : {metrics.box.map*100:.2f}%')
print(f'  Précision    : {metrics.box.mp*100:.2f}%')
print(f'  Rappel       : {metrics.box.mr*100:.2f}%')
print(f'  Taille       : {SIZE_MB:.1f} MB')
print(f'  Paramètres   : {PARAMS:.2f} M')
print(f'  Temps train  : {TRAIN_TIME/3600:.2f}h')
print(f'{"═"*60}')
print(f'\n  Par classe :')
for i, name in enumerate(CLASS_NAMES):
    try:
        ap = metrics.box.ap50[i] * 100
        print(f'    {i} - {name:20s} : {ap:.2f}%')
    except:
        print(f'    {i} - {name:20s} : N/A')

# Sauvegarder modèle sur Drive
shutil.copy(BEST_PT, f'{RESULTS_DIR}/{MODEL_NAME}_best.pt')
print(f'\n  💾 best.pt → Drive ✅')

# Sauvegarder figures
for fig in ['results.png','confusion_matrix.png','PR_curve.png','F1_curve.png']:
    src = f'{WORK_DIR}/runs/{MODEL_NAME}/{fig}'
    if os.path.exists(src):
        shutil.copy(src, f'{RESULTS_DIR}/{MODEL_NAME}_{fig}')
        print(f'  💾 {fig} → Drive ✅')

In [ ]:
# ═══════════════════════════════════════════════════
# CELLULE 9 — Sauvegarde CSV cumulatif
# ═══════════════════════════════════════════════════
import pandas as pd, os

# Résultats par classe
class_ap = {}
for i, name in enumerate(CLASS_NAMES):
    try:
        class_ap[name] = round(float(metrics.box.ap50[i]) * 100, 2)
    except:
        class_ap[name] = 0.0

result = {
    'Model'              : MODEL_NAME,
    'mAP@50 (%)'         : round(metrics.box.map50 * 100, 2),
    'mAP@50-95 (%)'      : round(metrics.box.map * 100, 2),
    'Precision (%)'      : round(metrics.box.mp * 100, 2),
    'Recall (%)'         : round(metrics.box.mr * 100, 2),
    'Size (MB)'          : round(SIZE_MB, 1),
    'Params (M)'         : round(PARAMS, 2),
    'Train time (h)'     : round(TRAIN_TIME / 3600, 2),
    'bus AP (%)'         : class_ap.get('bus', 0),
    'car AP (%)'         : class_ap.get('car', 0),
    'emergency AP (%)'   : class_ap.get('emergency_vehicle', 0),
    'motorcycle AP (%)'  : class_ap.get('motorcycle', 0),
    'person AP (%)'      : class_ap.get('person', 0),
    'truck AP (%)'       : class_ap.get('truck', 0),
    'cls_pw'             : CLS_PW,
    'epochs_done'        : EPOCHS,
}

# Charger ou créer CSV
CSV_PATH = f'{RESULTS_DIR}/benchmark_results.csv'
if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    df = df[df['Model'] != MODEL_NAME]  # Supprimer si déjà existant
    df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
else:
    df = pd.DataFrame([result])

df = df.sort_values('mAP@50 (%)', ascending=False)
df.to_csv(CSV_PATH, index=False)

print('\n📊 TABLEAU BENCHMARK CUMULATIF')
print('═'*80)
cols = ['Model','mAP@50 (%)','mAP@50-95 (%)','Precision (%)','Recall (%)','Size (MB)','Params (M)']
print(df[cols].to_string(index=False))
print('═'*80)
print(f'\n💾 CSV → {CSV_PATH} ✅')

In [ ]:
# ═══════════════════════════════════════════════════
# CELLULE 10 — Export ONNX (pour Pi 5)
# ═══════════════════════════════════════════════════
from ultralytics import YOLO
import shutil, glob, os

model_export = YOLO(f'{RESULTS_DIR}/{MODEL_NAME}_best.pt')
model_export.export(
    format   = 'onnx',
    imgsz    = 640,
    simplify = True,
    opset    = 12,
)

# Trouver ONNX généré
onnx_files = glob.glob(f'{RESULTS_DIR}/**/*.onnx', recursive=True)
if not onnx_files:
    onnx_files = glob.glob('/content/**/*.onnx', recursive=True)

if onnx_files:
    dst = f'{RESULTS_DIR}/{MODEL_NAME}_best.onnx'
    if os.path.abspath(onnx_files[0]) != os.path.abspath(dst):
        shutil.copy(onnx_files[0], dst)
    size_onnx = os.path.getsize(dst) / 1e6
    print(f'✅ ONNX exporté : {MODEL_NAME}_best.onnx ({size_onnx:.1f} MB)')
    print(f'💾 → {RESULTS_DIR}')

print(f'\n🎉 Session {MODEL_NAME} terminée !')
print(f'\nProchaine session → modifier MODEL_NAME dans cellule 6')